# Homework 1: Exploratory Data Analysis and Data Munging

MSE 125 — Spring 2026

## Setup

This homework covers Lectures 1–3: exploratory data analysis,
visualization, and data munging. You will work with H-1B visa
application data from the U.S. Department of Labor (FY2024). Every year,
employers file Labor Condition Applications (LCAs) to sponsor workers
for H-1B visas — each row in this dataset is one such filing.

The [Office of Foreign Labor
Certification](https://www.dol.gov/agencies/eta/foreign-labor/performance)
publishes the data. The setup cell below downloads it directly from the
DOL website. The file is about 80 MB, so the download may take a minute.
If the download fails, ask a TA for a local copy.

You are encouraged to use AI coding assistants (Claude Code, Copilot,
ChatGPT) throughout this homework. Problem 4 specifically asks you to
document how AI handles the data.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

In [2]:
# Download H-1B LCA disclosure data (FY2024 Q4) from the Department of Labor
import os, urllib.request

DATA_URL = "https://www.dol.gov/sites/dolgov/files/ETA/oflc/pdfs/LCA_Disclosure_Data_FY2024_Q4.xlsx"
LOCAL_FILE = "LCA_Disclosure_Data_FY2024_Q4.xlsx"

if not os.path.exists(LOCAL_FILE):
    print("Downloading H-1B data from DOL (~80 MB, may take a minute)...")
    urllib.request.urlretrieve(DATA_URL, LOCAL_FILE)
    print("Download complete.")

# Select the columns we need (the full file has 97 columns)
USE_COLS = [
    'CASE_NUMBER', 'CASE_STATUS', 'RECEIVED_DATE', 'DECISION_DATE',
    'VISA_CLASS', 'JOB_TITLE', 'SOC_CODE', 'SOC_TITLE',
    'FULL_TIME_POSITION', 'BEGIN_DATE', 'END_DATE',
    'EMPLOYER_NAME', 'EMPLOYER_CITY', 'EMPLOYER_STATE', 'EMPLOYER_POSTAL_CODE',
    'NAICS_CODE', 'AGENT_REPRESENTING_EMPLOYER',
    'SECONDARY_ENTITY', 'SECONDARY_ENTITY_BUSINESS_NAME',
    'WORKSITE_CITY', 'WORKSITE_STATE', 'WORKSITE_POSTAL_CODE',
    'WAGE_RATE_OF_PAY_FROM', 'WAGE_RATE_OF_PAY_TO', 'WAGE_UNIT_OF_PAY',
    'PREVAILING_WAGE', 'PW_UNIT_OF_PAY', 'PW_WAGE_LEVEL',
    'H_1B_DEPENDENT', 'WILLFUL_VIOLATOR', 'SUPPORT_H1B', 'STATUTORY_BASIS',
]

h1b = pd.read_excel(LOCAL_FILE, engine='openpyxl', usecols=USE_COLS)
print(f"Loaded {len(h1b):,} rows x {len(h1b.columns)} columns")

Loaded 120,897 rows x 32 columns

Each row is one LCA filing. The dataset contains about 120,000 filings
covering employer information, job details, wages, and work locations.

### Submission

Submit your completed Jupyter notebook (`.ipynb`) on Gradescope. Keep
all code cells and their output visible. Write interpretations in
markdown cells, not code comments.

------------------------------------------------------------------------

## Problem 1: First look

**Part (a):** How many rows and columns does the dataset have? Print the
first few rows and examine the column names. Which columns relate to
wages? Identify two columns whose pandas `dtype` does not match their
real-world meaning (e.g., a column stored as a number that is really a
category, or a column stored as a string that should be a date).

**Part (b):** Before computing anything, write down the range of values
you would expect to see in this column if it contained annual salaries.
Then summarize `WAGE_RATE_OF_PAY_FROM` (mean, median, min, max, standard
deviation). Do the numbers match your expectations? Identify at least
two things that look suspicious, and explain why they caught your
attention.

**Part (c):** Examine the `WAGE_UNIT_OF_PAY` column. How many distinct
values does it contain, and what fraction of rows falls in each
category? Explain how this column relates to the suspicious statistics
you found in part (b).

------------------------------------------------------------------------

## Problem 2: Cleaning the wage data

A journalist asks you: *“What does the typical H-1B worker earn?”* To
answer this question, you need a clean annual salary for every filing.

**Part (a):** Create a new column called `annual_wage` that converts all
wages to an annual basis. Use the standard conversion factors: hourly ×
2,080 (40 hrs/wk × 52 wks), weekly × 52, biweekly × 26, monthly × 12,
and yearly wages as-is. How many rows have each wage unit?

*Hint:* You can use
`df['WAGE_UNIT_OF_PAY'].map({'Year': 1, 'Hour': 2080, ...})` to look up
the conversion factor for each row, then multiply.

**Part (b):** Print the summary statistics for `annual_wage`. Are there
still problems? Identify any remaining outliers — propose reasonable
upper and lower thresholds and justify your choices. How many rows do
your thresholds flag? Choose a strategy for handling them (remove, cap,
or something else) and explain why.

**Part (c):** Make two plots: (1) a histogram of `annual_wage` before
outlier removal and (2) a histogram after your cleaning. Use appropriate
axis labels, titles, and bin widths. In 2–3 sentences, describe the
shape of the cleaned wage distribution (center, spread, skewness) and
report the median annual wage.

**Part (d):** Answer the journalist’s question in one paragraph. Report
the median wage and briefly explain why the median is more appropriate
than the mean here. Mention any caveats about what this number does and
does not represent (e.g., these are *offered* wages for visa applicants,
not all U.S. workers).

**Note for later problems:** Use your cleaned `annual_wage` (with
outliers removed) for all wage summaries and comparisons. Use all
filings for counts unless a problem says otherwise.

------------------------------------------------------------------------

## Problem 3: Employer analysis

**Part (a):** Which employers file the most H-1B applications? Display
the top 15 by filing count. Look carefully at the names — do any
employers appear more than once under slightly different names? Identify
at least one case. (Look at capitalization and spacing.)

**Part (b):** Write code to standardize the employer names you
identified in part (a). A simple approach suffices (e.g.,
`.str.upper().str.strip()`). After cleaning, recompute the top 15 and
display the updated table. Did the rankings change? Identify one case
where it is ambiguous whether two similar names refer to the same
employer (e.g., names that differ by “LLC” vs. “Inc”). What additional
information would you need to decide whether to merge them?

**Part (c):** Create a horizontal bar chart showing the top 15 employers
by filing count (after cleaning). For each employer, also show the
median annual wage (use the cleaned wage from Problem 2). You may use
two subplots, a color encoding, or annotations — choose whatever makes
the comparison clear.

**Part (d):** An immigration policy analyst claims that consulting firms
are “gaming” the H-1B system by filing high volumes of applications at
below-market wages. Based on your chart, is this characterization fair?
What does the data support, what does it not support, and what
additional information would you need to evaluate the claim?

------------------------------------------------------------------------

## Problem 4: AI-assisted analysis

Open an AI assistant (Claude, ChatGPT, Copilot, or another tool of your
choice). Paste the output of the following into your chat, then ask the
AI: *“Which states offer the highest H-1B salaries? Show me the top 10
with a bar chart.”*

In [3]:
# Run this cell, then copy the printed output into your AI assistant
print(h1b.dtypes.to_string())
print("\n---\n")
print(h1b.head(20).to_string())

CASE_NUMBER                                  str
CASE_STATUS                                  str
RECEIVED_DATE                     datetime64[us]
DECISION_DATE                     datetime64[us]
VISA_CLASS                                   str
JOB_TITLE                                    str
SOC_CODE                                     str
SOC_TITLE                                    str
FULL_TIME_POSITION                           str
BEGIN_DATE                        datetime64[us]
END_DATE                          datetime64[us]
EMPLOYER_NAME                                str
EMPLOYER_CITY                             object
EMPLOYER_STATE                               str
EMPLOYER_POSTAL_CODE                         str
NAICS_CODE                                 int64
AGENT_REPRESENTING_EMPLOYER                  str
SECONDARY_ENTITY                             str
SECONDARY_ENTITY_BUSINESS_NAME               str
WORKSITE_CITY                             object
WORKSITE_STATE      

**Part (a):** Copy the AI’s response (code and analysis) into your
notebook and run the code. Does the analysis produce correct results?
Identify at least two specific errors, questionable choices, or missing
steps in the AI’s output. For each issue, explain: (i) what went wrong,
(ii) why someone skimming the output might not notice, and (iii) how to
fix it. (Hint: think about what you learned in Problems 1–2, and check
which filings the AI included.)

*Grading note:* Different AI tools produce different outputs — we will
grade on the quality of your diagnosis and correction, not on matching a
specific bug list.

**Part (b):** Produce your own corrected version of the analysis.
Compare your result to the AI’s. How did the errors affect the
conclusions?

**Part (c):** Describe when the AI was most helpful and when human
judgment was necessary. What checks would you recommend running any time
you use AI on a new dataset?

------------------------------------------------------------------------

## Problem 5: Missing data and joins

A colleague proposes dropping all rows with any missing data before
analyzing the H-1B filings. You want to check whether that would bias
the results.

**Part (a):** How much data is missing? For each column, report the
number and percentage of null values. Which columns have the most
missing data? Pick two columns with moderate-to-high missingness and
describe, in one sentence each, a plausible reason why those values
might be missing.

**Part (b):** Choose one column with meaningful missingness (e.g.,
`SUPPORT_H1B`, `PW_WAGE_LEVEL`, or `STATUTORY_BASIS`). Split the data
into two groups: rows where the column is missing and rows where it is
present. Compare the `annual_wage` between the two groups using a
histogram or density plot and a summary statistic (e.g., means or
medians). Is there a notable difference?

**Part (c):** Based on your findings in part (b), do you think the
missing values in the column you chose are MCAR (missing completely at
random), MAR (missing at random, i.e., related to other observed
variables), or MNAR (missing not at random)? There may not be a single
correct answer — justify your reasoning in 2–3 sentences. What would go
wrong if you simply dropped all rows with missing values before
computing summary statistics?

**Part (d):** The `SOC_CODE` column contains Standard Occupational
Classification codes (e.g., `15-1252.00` for “Software Developers”). The
Bureau of Labor Statistics publishes national median wages by SOC code.
The cell below loads a small reference table using 6-digit codes
(without the `.00` suffix).

In [4]:
# BLS national median wages by SOC code (2023 OES survey, selected codes)
bls_wages = pd.DataFrame({
    'SOC_6': ['15-1252', '15-1211', '15-1299', '15-1244', '15-1232',
              '11-3021', '13-1111', '13-2011', '15-1241', '15-1256'],
    'SOC_TITLE_BLS': ['Software Developers', 'Computer Systems Analysts',
                      'Computer Occupations, All Other', 'Network/Systems Administrators',
                      'Computer User Support Specialists', 'Computer and IS Managers',
                      'Management Analysts', 'Accountants and Auditors',
                      'Computer Network Architects', 'Software Quality Assurance Analysts'],
    'national_median_wage': [127260, 99270, 97430, 90520, 57890,
                             164070, 99410, 79880, 126900, 98220],
})

A common critique of the H-1B program is that employers use it to hire
foreign workers at wages below the domestic market rate. Join the H-1B
filings with `bls_wages` to test this claim. (Hint: compare one
`SOC_CODE` value to one `SOC_6` value to see what format mismatch you
need to handle.) How many filings match a BLS code? For the matched
rows, compare the H-1B `annual_wage` to the `national_median_wage` in a
table or plot. Do H-1B filings tend to offer more or less than the
national median for the same occupation? Does the answer differ across
job categories? What caveats would you attach before drawing a policy
conclusion?